# Бонус: Рисование черепашкой через Model Context Protocol

В этом ноутбуке мы используем **Model Context Protocol (MCP)** для управления черепашьей графикой. Этот пример показывает, как можно вызывать MCP-сервер, работающий на локальном компьютере. К сожалению, Responses API мало помогает в этой задаче, и вызов функций приходится обрабатывать локально - из-за этого код получается достаточно сложным.

> Поскольку в примере мы взаимодействуем с сервером на локальном компьютере, то запустить код в Jupyter Notebook на удалённом сервере не получится. Вам необходимо локальное Python окружение с поддержкой Jupyter. 

## Архитектура

```
┌─────────────────┐     MCP/HTTP      ┌─────────────────┐
│  Jupyter Client │ ───────────────▶  │  Turtle Server  │
│  (этот ноутбук) │                   │  (Python turtle)│
└─────────────────┘                   └─────────────────┘
        │                                     │
        ▼                                     ▼
    LLM (Qwen3)                       Окно с рисунком
```

## Подготовка

**Перед запуском этого ноутбука** необходимо запустить MCP-сервер в отдельном окне терминала:

```bash
python turtle_mcp_server.py
```

Сервер будет доступен по адресу `http://localhost:8000/mcp`.

In [ ]:
%pip install --upgrade openai python-dotenv mcp

**ВНИМАНИЕ**: После установки библиотек рекомендуется перезапустить Kernel ноутбука.

In [ ]:
!curl -o .env {{url_of_dotenv_file}}\n

In [1]:
import os
import json
import asyncio
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

folder_id = os.environ["folder_id"]
api_key = os.environ["api_key"]

def printx(string):
    display(Markdown(string))

print(f"✅ Авторизация настроена (folder_id: {folder_id[:8]}...)")

✅ Авторизация настроена (folder_id: b1gbicod...)


---

## Часть 1: Подключение к MCP-серверу

Используем MCP SDK для подключения к серверу и получения списка доступных инструментов:

In [5]:
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client

MCP_SERVER_URL = "http://localhost:8000/mcp"

async def list_mcp_tools():
    """Подключается к MCP-серверу и выводит список инструментов."""
    async with streamable_http_client(MCP_SERVER_URL) as (read, write, _):
        async with ClientSession(read, write) as session:
            await session.initialize()
            tools = await session.list_tools()
            
            print("🐢 Доступные инструменты MCP-сервера:")
            for tool in tools.tools:
                print(f"  • {tool.name}: {tool.description}")
            
            return tools.tools

# Запускаем проверку подключения
await list_mcp_tools()

🐢 Доступные инструменты MCP-сервера:
  • forward: Переместить черепашку вперёд на указанное расстояние в пикселях.
При опущенном пере оставляет линию.

Args:
    distance: Расстояние в пикселях (обычно от 10 до 200)
  • left: Повернуть черепашку налево на указанный угол в градусах.

Args:
    degrees: Угол поворота в градусах (например, 90 для прямого угла)
  • right: Повернуть черепашку направо на указанный угол в градусах.

Args:
    degrees: Угол поворота в градусах (например, 90 для прямого угла)
  • penup: Поднять перо. После этого черепашка будет двигаться без рисования линий.
Используйте для перемещения в другую точку без рисования.
  • pendown: Опустить перо. После этого черепашка будет рисовать линии при движении.
  • reset_canvas: Очистить холст и вернуть черепашку в начальную позицию.
Используйте перед началом нового рисунка.
  • get_position: Получить текущую позицию и направление черепашки.


[Tool(name='forward', title=None, description='Переместить черепашку вперёд на указанное расстояние в пикселях.\nПри опущенном пере оставляет линию.\n\nArgs:\n    distance: Расстояние в пикселях (обычно от 10 до 200)', inputSchema={'additionalProperties': False, 'properties': {'distance': {'type': 'number'}}, 'required': ['distance'], 'type': 'object'}, outputSchema={'properties': {'result': {'type': 'string'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-wrap-result': True}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None),
 Tool(name='left', title=None, description='Повернуть черепашку налево на указанный угол в градусах.\n\nArgs:\n    degrees: Угол поворота в градусах (например, 90 для прямого угла)', inputSchema={'additionalProperties': False, 'properties': {'degrees': {'type': 'number'}}, 'required': ['degrees'], 'type': 'object'}, outputSchema={'properties': {'result': {'type': 'string'}}, 'required': ['result'], 'type': 'object', 'x-fastmcp-

---

## Часть 2: Настройка OpenAI клиента

Создадим клиент для работы с Yandex Cloud:

In [6]:
from openai import OpenAI

model = f"gpt://{folder_id}/qwen3-235b-a22b-fp8/latest"

client = OpenAI(
    base_url="https://ai.api.cloud.yandex.net/v1",
    api_key=api_key,
    project=folder_id
)

print("✅ Клиент OpenAI настроен")

✅ Клиент OpenAI настроен


---

## Часть 3: Класс Agent с поддержкой MCP

Создадим класс Agent, который умеет работать как с локальными функциями (Pydantic), так и с MCP-серверами:

In [7]:
from pydantic import BaseModel
from mcp import ClientSession
from mcp.client.streamable_http import streamable_http_client
from typing import Optional


class MCPAgent:
    """
    Агент с поддержкой Function Calling через MCP.
    Поддерживает как MCP-инструменты, так и локальные Pydantic-функции.
    """
    
    def __init__(
        self, 
        instruction: str,
        mcp_url: Optional[str] = None,
        local_tools: list = None,
        model: str = model,
        max_iterations: int = 50
    ):
        self.instruction = instruction
        self.model = model
        self.max_iterations = max_iterations
        self.mcp_url = mcp_url
        
        # Локальные инструменты (Pydantic)
        self.local_tools = local_tools or []
        self.local_tool_map = {cls.__name__: cls for cls in self.local_tools}
        
        # MCP-инструменты (будут загружены при первом вызове)
        self.mcp_tools = []
        self.mcp_tool_names = set()
        
        # Сессии
        self.sessions = {}
    
    def _get_local_tools_schema(self) -> list:
        """Генерирует схему для локальных инструментов."""
        return [
            {
                "type": "function",
                "name": cls.__name__,
                "description": cls.__doc__ or "",
                "parameters": cls.model_json_schema()
            }
            for cls in self.local_tools
        ]
    
    def _mcp_tool_to_schema(self, tool) -> dict:
        """Конвертирует MCP-инструмент в схему OpenAI."""
        return {
            "type": "function",
            "name": tool.name,
            "description": tool.description or "",
            "parameters": tool.inputSchema if tool.inputSchema else {"type": "object", "properties": {}}
        }
    
    async def _load_mcp_tools(self, session: ClientSession):
        """Загружает список инструментов с MCP-сервера."""
        tools_result = await session.list_tools()
        self.mcp_tools = [self._mcp_tool_to_schema(t) for t in tools_result.tools]
        self.mcp_tool_names = {t.name for t in tools_result.tools}
    
    async def _call_mcp_tool(self, session: ClientSession, name: str, args: dict) -> str:
        """Вызывает инструмент на MCP-сервере."""
        result = await session.call_tool(name, arguments=args)
        # Извлекаем текстовый результат
        if result.content:
            for content in result.content:
                if hasattr(content, 'text'):
                    return content.text
        return str(result)
    
    def _call_local_tool(self, name: str, args: dict) -> str:
        """Вызывает локальный инструмент."""
        cls = self.local_tool_map[name]
        obj = cls.model_validate(args)
        return obj.execute()
    
    async def __call__(self, message: str, session_id: str = "default", verbose: bool = True) -> str:
        """
        Обрабатывает сообщение пользователя с поддержкой MCP и локальных инструментов.
        """
        if not self.mcp_url:
            raise ValueError("MCP URL не задан")
        
        async with streamable_http_client(self.mcp_url) as (read, write, _):
            async with ClientSession(read, write) as mcp_session:
                await mcp_session.initialize()
                
                # Загружаем MCP-инструменты
                await self._load_mcp_tools(mcp_session)
                
                # Объединяем все инструменты
                all_tools = self.mcp_tools + self._get_local_tools_schema()
                
                if verbose:
                    print(f"📦 Загружено инструментов: {len(all_tools)} (MCP: {len(self.mcp_tools)}, локальных: {len(self.local_tools)})")
                
                # Получаем или создаём сессию
                session = self.sessions.get(session_id, {"last_response_id": None})
                
                # Первый запрос к LLM
                res = client.responses.create(
                    model=self.model,
                    tools=all_tools,
                    instructions=self.instruction,
                    previous_response_id=session["last_response_id"],
                    store=True,
                    input=message
                )
                
                # Цикл обработки функций
                for _ in range(self.max_iterations):
                    tool_calls = [item for item in res.output if item.type == "function_call"]
                    
                    if not tool_calls:
                        break
                    
                    # Обрабатываем вызовы
                    outputs = []
                    for call in tool_calls:
                        args = json.loads(call.arguments) if call.arguments else {}
                        
                        if verbose:
                            print(f"  🔧 {call.name}({args})")
                        
                        try:
                            # Определяем, MCP это или локальный инструмент
                            if call.name in self.mcp_tool_names:
                                result = await self._call_mcp_tool(mcp_session, call.name, args)
                            elif call.name in self.local_tool_map:
                                result = self._call_local_tool(call.name, args)
                            else:
                                result = f"Неизвестный инструмент: {call.name}"
                        except Exception as e:
                            result = f"Ошибка: {e}"
                        
                        if verbose:
                            print(f"    → {result}")
                        
                        outputs.append({
                            "type": "function_call_output",
                            "call_id": call.call_id,
                            "output": result
                        })
                    
                    # Отправляем результаты LLM
                    res = client.responses.create(
                        model=self.model,
                        tools=all_tools,
                        previous_response_id=res.id,
                        store=True,
                        input=outputs
                    )
                
                # Сохраняем состояние сессии
                session["last_response_id"] = res.id
                self.sessions[session_id] = session
                
                return res.output_text or "Готово!"


print("✅ Класс MCPAgent определён")

✅ Класс MCPAgent определён


---

## Часть 4: Создание агента-художника

Теперь создадим агента, который будет управлять черепашкой через MCP:

In [8]:
artist_instruction = """
Ты — талантливый художник, который рисует с помощью черепашьей графики через MCP-сервер.

Черепашка начинает в центре холста и смотрит вправо (на восток).
Ты можешь управлять ей с помощью MCP-инструментов:
- forward(distance) — движение вперёд на distance пикселей
- left(degrees) — поворот налево на degrees градусов
- right(degrees) — поворот направо на degrees градусов
- penup() — поднять перо (перемещение без рисования)
- pendown() — опустить перо (рисовать при движении)
- reset_canvas() — очистить холст и начать заново
- get_position() — узнать текущую позицию черепашки

Когда пользователь просит что-то нарисовать:
1. Сначала вызови reset_canvas() чтобы очистить холст
2. Продумай, как разбить рисунок на простые линии
3. Вызывай команды по очереди для создания рисунка
4. Используй размеры 50-150 пикселей для хороших пропорций
5. Для сложных фигур используй penup/pendown для перемещения без рисования

Рисуй аккуратно и старайся создать узнаваемые фигуры!
"""

# Создаём агента с подключением к MCP-серверу
artist = MCPAgent(
    instruction=artist_instruction,
    mcp_url="http://localhost:8000/mcp"
)

print("🎨 Агент-художник готов к работе!")

🎨 Агент-художник готов к работе!


---

## Часть 5: Рисуем с помощью агента

Теперь можем давать агенту задания на рисование!

In [9]:
async def draw(prompt: str):
    """Попросить агента нарисовать что-то через MCP."""
    print(f"🎨 Задание: {prompt}")
    print("=" * 60)
    
    result = await artist(prompt)
    
    print("=" * 60)
    printx(result)

### Примеры рисования

In [10]:
# Рисуем квадрат
await draw("Нарисуй квадрат со стороной 100 пикселей")

🎨 Задание: Нарисуй квадрат со стороной 100 пикселей
📦 Загружено инструментов: 7 (MCP: 7, локальных: 0)
  🔧 reset_canvas({})
    → Холст очищен, черепашка в начальной позиции
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей


Квадрат со стороной 100 пикселей успешно нарисован. Черепашка вернулась в исходное положение и завершила рисование.

In [11]:
# Рисуем дом
await draw("Нарисуй простой дом: квадрат с треугольной крышей сверху")

🎨 Задание: Нарисуй простой дом: квадрат с треугольной крышей сверху
📦 Загружено инструментов: 7 (MCP: 7, локальных: 0)
  🔧 reset_canvas({})
    → Холст очищен, черепашка в начальной позиции
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 left({'degrees': 30})
    → Черепашка повернулась налево на 30.0 град

Дом нарисован: сначала был нарисован квадрат (основание дома), затем от верхнего левого угла квадрата была построена треугольная крыша. Крыша выполнена как равносторонний треугольник, где каждый угол составляет 60 градусов, а повороты корректировались соответствующим образом. Рисунок завершён.

In [12]:
# Рисуем дерево
await draw("Нарисуй простое дерево: вертикальный ствол и треугольную крону")

🎨 Задание: Нарисуй простое дерево: вертикальный ствол и треугольную крону
📦 Загружено инструментов: 7 (MCP: 7, локальных: 0)
  🔧 reset_canvas({})
    → Холст очищен, черепашка в начальной позиции
  🔧 forward({'distance': 100})
    → Черепашка продвинулась вперёд на 100.0 пикселей
  🔧 penup({})
    → Перо поднято — черепашка не будет рисовать
  🔧 right({'degrees': 90})
    → Черепашка повернулась направо на 90.0 градусов
  🔧 forward({'distance': 30})
    → Черепашка продвинулась вперёд на 30.0 пикселей
  🔧 left({'degrees': 90})
    → Черепашка повернулась налево на 90.0 градусов
  🔧 pendown({})
    → Перо опущено — черепашка будет рисовать
  🔧 left({'degrees': 30})
    → Черепашка повернулась налево на 30.0 градусов
  🔧 forward({'distance': 80})
    → Черепашка продвинулась вперёд на 80.0 пикселей
  🔧 left({'degrees': 120})
    → Черепашка повернулась налево на 120.0 градусов
  🔧 forward({'distance': 80})
    → Черепашка продвинулась вперёд на 80.0 пикселей
  🔧 left({'degrees': 120})
  

Дерево нарисовано: сначала был нарисован вертикальный ствол длиной 100 пикселей. Затем черепашка подняла перо, переместилась в точку основания кроны, опустила перо и нарисовала треугольную крону (равносторонний треугольник со стороной 80 пикселей). Рисунок завершён.

In [13]:
# Рисуем звезду
await draw("Нарисуй пятиконечную звезду")

🎨 Задание: Нарисуй пятиконечную звезду
📦 Загружено инструментов: 7 (MCP: 7, локальных: 0)
  🔧 reset_canvas({})
    → Холст очищен, черепашка в начальной позиции
  🔧 left({'degrees': 72})
    → Черепашка повернулась налево на 72.0 градусов
  🔧 forward({'distance': 150})
    → Черепашка продвинулась вперёд на 150.0 пикселей
  🔧 right({'degrees': 144})
    → Черепашка повернулась направо на 144.0 градусов
  🔧 forward({'distance': 150})
    → Черепашка продвинулась вперёд на 150.0 пикселей
  🔧 right({'degrees': 144})
    → Черепашка повернулась направо на 144.0 градусов
  🔧 forward({'distance': 150})
    → Черепашка продвинулась вперёд на 150.0 пикселей
  🔧 right({'degrees': 144})
    → Черепашка повернулась направо на 144.0 градусов
  🔧 forward({'distance': 150})
    → Черепашка продвинулась вперёд на 150.0 пикселей
  🔧 right({'degrees': 144})
    → Черепашка повернулась направо на 144.0 градусов
  🔧 forward({'distance': 150})
    → Черепашка продвинулась вперёд на 150.0 пикселей


Пятиконечная звезда успешно нарисована. Черепашка начала с поворота на 72 градуса для правильного направления, затем нарисовала пять линий длиной 150 пикселей, поворачиваясь после каждой на 144 градуса направо, чтобы сформировать характерные острые углы звезды. Рисунок завершён.

---

## Заключение

В этом ноутбуке мы создали систему рисования через **Model Context Protocol (MCP)**:

### Преимущества MCP:
- 🔌 **Стандартизация** — единый протокол для всех инструментов
- 🌐 **Удалённый доступ** — сервер может работать на другой машине
- 🔒 **Изоляция** — инструменты работают в отдельном процессе
- 🔄 **Переиспользование** — один сервер для многих клиентов

### Возможные улучшения:
- Добавить цвета и толщину линий
- Попробовать рисовать более сложные рисунки и посмотреть, как можно улучшить промпт, чтобы модель справлялась
- Попробовать визуальную модель и поэкспериментировать, может ли она нарисовать портрет человека по фотографии